In [2]:
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ================= 配置区域 =================
# 1. 输出目录
DATASET_ROOT = "./dataset_zero_cfo_rad" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

# 2. 真实数据映射
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

# 3. 类别与参数
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800 # 每类生成的样本数 (Train+Val)
TRAIN_VAL_RATIO = 0.8             # 80% 训练, 20% 验证
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 1. R+A+D 特征生成 (3通道) =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_rad_heatmap(iq_complex_data, img_size=64):
    if iq_complex_data.dim() == 1: iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    if iq_data.numel() < 100: return torch.zeros(3, img_size, img_size)

    # 归一化
    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    # R 通道 (I vs Q)
    hist_r = fast_hist2d(norm_iq.real, norm_iq.imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    # A 通道 (Phase vs Mag)
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    hist_a = fast_hist2d(phase, mag, bins=img_size, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    
    # D 通道 (Differential)
    diff = norm_iq[1:] * torch.conj(norm_iq[:-1])
    d_scale = torch.sqrt(torch.mean(diff.real**2 + diff.imag**2)) + 1e-9
    diff = diff / (3.0 * d_scale)
    hist_d = fast_hist2d(diff.real, diff.imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    def process(h):
        h = torch.log1p(h)
        return (h - h.min()) / (h.max() - h.min() + 1e-8)
        
    img_tensor = torch.stack([process(hist_r), process(hist_a), process(hist_d)], dim=0) 
    return torch.flip(img_tensor, [1]) 

def save_rgb(tensor, path):
    tensor = tensor.detach().cpu()
    arr = (tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    Image.fromarray(arr, mode='RGB').save(path)

# ================= 2. 信号仿真 (无频偏版) =================

def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    
    # [关键] 强制频偏为 0 (Zero CFO)
    # 我们只保留随机初始相位，但不让它旋转
    phase_offset = np.random.uniform(0, 2*np.pi)
    
    # [修正处] 使用 np.exp 计算标量，然后转为 tensor，或者直接利用 broadcasting
    # 这里使用 torch.tensor 包装 np.exp 的结果，确保类型兼容
    rotation = torch.tensor(np.exp(1j * phase_offset), dtype=torch.complex64)
    
    sig = sig * rotation
    
    # 其他微量损伤保留，避免过度理想化
    phase_jitter = torch.randn(N) * 0.002
    sig = sig * torch.exp(1j * phase_jitter)
    
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

# ... (辅助函数 get_32qam, get_apsk 保持不变) ...
def get_32qam_constellation():
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            if not (abs(x) == 5 and abs(y) == 5): points.append(x + 1j*y)
    return torch.tensor(points, dtype=torch.complex64)

def get_apsk_symbols(M, ratios, num_symbols):
    if M == 16:
        R1, R2 = 1.0, ratios[0]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_outer = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        constellation = torch.cat([pts_inner, pts_outer])
    elif M == 32:
        R1, R2, R3 = 1.0, ratios[0], ratios[1]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_mid = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        pts_outer = R3 * torch.exp(1j * (torch.arange(16)*2*np.pi/16))
        constellation = torch.cat([pts_inner, pts_mid, pts_outer])
    indices = torch.randint(0, M, (num_symbols,))
    return constellation[indices]

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    syms = None
    if mod_type == "BPSK": syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK": syms = torch.exp(1j * (np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2))
    elif mod_type == "8PSK": syms = torch.exp(1j * torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8)
    elif mod_type == "OQPSK": syms = torch.exp(1j * (np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2))
    elif mod_type == "PI4DQPSK": 
        bits = torch.randint(0, 4, (num_symbols,))
        phase_shifts = torch.tensor([np.pi/4, 3*np.pi/4, -np.pi/4, -3*np.pi/4])[bits]
        syms = torch.exp(1j * torch.cumsum(phase_shifts, dim=0))
    elif mod_type == "2ASK": syms = torch.randint(0, 2, (num_symbols,)).float() + 0j
    elif mod_type == "16QAM": 
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "32QAM": syms = get_32qam_constellation()[torch.randint(0, 32, (num_symbols,))]
    elif mod_type == "64QAM": 
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "16APSK": syms = get_apsk_symbols(16, [2.7], num_symbols)
    elif mod_type == "32APSK": syms = get_apsk_symbols(32, [2.6, 4.5], num_symbols)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return apply_channel_effects(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    if mod_type == "OQPSK":
        upsampled_i = torch.zeros(num_symbols * sps); upsampled_i[::sps] = syms.real
        q_temp = torch.zeros(num_symbols * sps); q_temp[::sps] = syms.imag
        upsampled_q = torch.roll(q_temp, shifts=sps//2); upsampled_q[:sps//2] = 0
        upsampled = upsampled_i + 1j * upsampled_q
    else: upsampled[::sps] = syms

    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    
    return apply_channel_effects(sig, snr_db)

# ================= 3. 执行生成 =================

if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT):
        print("Cleaning old dataset...")
        shutil.rmtree(DATASET_ROOT)
    
    print("===== 1. Generating ZERO CFO Simulation (R+A+D) =====")
    
    for cls_name in CLASS_NAMES:
        # 创建目录
        path_train = os.path.join(SIM_TRAIN_DIR, cls_name)
        path_val = os.path.join(SIM_VAL_DIR, cls_name)
        os.makedirs(path_train, exist_ok=True)
        os.makedirs(path_val, exist_ok=True)
        
        print(f"Generating {cls_name}...")
        
        # 1. 生成数据 (Total = TOTAL_SIM_SAMPLES_PER_CLASS)
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            snr = np.random.uniform(-5, 25)
            # 生成信号
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            heatmap = generate_rad_heatmap(sig.to(DEVICE), IMAGE_SIZE)
            
            # 随机划分 Train / Val
            if np.random.rand() < TRAIN_VAL_RATIO:
                save_path = os.path.join(path_train, f"{cls_name}_{i:04d}_{snr:.1f}.png")
            else:
                save_path = os.path.join(path_val, f"{cls_name}_{i:04d}_{snr:.1f}.png")
                
            save_rgb(heatmap, save_path)

    print("\n===== 2. Processing Real Data =====")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skip: {file_path}")
            continue
        print(f"Processing {label}...")
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q).astype(np.complex64) / 32767.0
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            for i in range(num_slices):
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                chunk_tensor = torch.from_numpy(chunk).to(DEVICE)
                heatmap = generate_rad_heatmap(chunk_tensor, IMAGE_SIZE)
                save_rgb(heatmap, os.path.join(save_dir, f"real_{i:04d}.png"))
        except Exception as e: print(e)
            
    print("\nData Generation Complete!")

Cleaning old dataset...
===== 1. Generating ZERO CFO Simulation (R+A+D) =====
Generating BPSK...


  0%|          | 0/800 [00:00<?, ?it/s]

100%|██████████| 800/800 [00:04<00:00, 198.17it/s]


Generating QPSK...


100%|██████████| 800/800 [00:02<00:00, 271.41it/s]


Generating 8PSK...


... (16 outputs left out)

100%|██████████| 800/800 [00:03<00:00, 217.98it/s]


Generating FSK2...


100%|██████████| 800/800 [00:02<00:00, 305.38it/s]



===== 2. Processing Real Data =====
Processing 8PSK...
Processing 16QAM...
Processing 64QAM...
Processing 2ASK...
Processing BPSK...
Processing QPSK...
Processing FSK2...
Processing OQPSK...
Processing PI4DQPSK...

Data Generation Complete!
